In [ ]:
```xml
<VSCode.Cell language="markdown">
# ☁️ Hybrid Cloud Architecture Guide

**Designing and managing applications across multiple cloud providers and on-premises environments**

This guide covers:
- Hybrid cloud patterns and topologies
- Multi-cloud strategy
- Data consistency across clouds
- Workload placement and optimization
- Cloud-agnostic applications
- Disaster recovery and failover
- Cost optimization across clouds
- Management and monitoring
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Hybrid Cloud Patterns

### Workload Classification
```python
from dataclasses import dataclass
from typing import Dict, List, Optional
from enum import Enum

class CloudPlacement(Enum):
    """Where workload should run"""
    ON_PREMISES = "on_premises"
    PUBLIC_CLOUD = "public_cloud"
    PRIVATE_CLOUD = "private_cloud"
    HYBRID = "hybrid"
    MULTI_CLOUD = "multi_cloud"

class WorkloadProfile:
    """Characteristics for workload placement"""
    
    def __init__(self, workload_name: str):
        self.workload_name = workload_name
        self.cpu_intensive: bool = False
        self.memory_intensive: bool = False
        self.data_sensitive: bool = False
        self.latency_critical: bool = False
        self.compliance_requirements: List[str] = []
        self.storage_requirements_gb: int = 0
        self.peak_concurrency: int = 0
        self.cost_sensitivity: str = "medium"  # low, medium, high
        self.has_gpu_requirement: bool = False
        self.scaling_needs: str = "moderate"  # none, moderate, aggressive

class WorkloadPlacementEngine:
    """Determine optimal placement for workloads"""
    
    def __init__(self):
        self.placement_policies: List[Dict] = []
        self.placement_history: Dict[str, CloudPlacement] = {}
    
    def add_policy(self, policy: Dict):
        """Add placement policy"""
        self.placement_policies.append(policy)
    
    def recommend_placement(self, profile: WorkloadProfile) -> CloudPlacement:
        """Recommend optimal placement"""
        
        score = {
            'on_premises': 0,
            'public_cloud': 0,
            'private_cloud': 0
        }
        
        # Data sensitivity → on-premises
        if profile.data_sensitive or 'HIPAA' in profile.compliance_requirements:
            score['on_premises'] += 10
            score['public_cloud'] -= 5
        
        # Latency critical → on-premises or private
        if profile.latency_critical:
            score['on_premises'] += 8
            score['private_cloud'] += 6
        
        # Cost sensitive + scaling → public cloud
        if profile.cost_sensitivity == 'high' and profile.scaling_needs == 'aggressive':
            score['public_cloud'] += 8
        
        # GPU requirements → public cloud
        if profile.has_gpu_requirement:
            score['public_cloud'] += 6
        
        # Memory intensive → public cloud (auto-scaling)
        if profile.memory_intensive:
            score['public_cloud'] += 5
        
        # Determine placement
        if score['on_premises'] > score['public_cloud'] and score['on_premises'] > score['private_cloud']:
            return CloudPlacement.ON_PREMISES
        elif score['public_cloud'] > score['on_premises'] and score['public_cloud'] > score['private_cloud']:
            return CloudPlacement.PUBLIC_CLOUD
        else:
            return CloudPlacement.PRIVATE_CLOUD

class HybridTopology:
    """Define hybrid cloud topology"""
    
    def __init__(self, topology_name: str):
        self.topology_name = topology_name
        self.regions: Dict[str, Dict] = {}  # region -> config
        self.connections: List[Dict] = []  # Links between regions
    
    def add_region(self, region_name: str, cloud_type: str, capacity: int):
        """Add region to topology"""
        self.regions[region_name] = {
            'cloud_type': cloud_type,
            'capacity': capacity,
            'utilization': 0
        }
    
    def add_connection(self, region1: str, region2: str, bandwidth_mbps: int, latency_ms: int):
        """Add connection between regions"""
        self.connections.append({
            'from': region1,
            'to': region2,
            'bandwidth': bandwidth_mbps,
            'latency': latency_ms,
            'redundant': False
        })
    
    def find_path_to_region(self, target_region: str, max_hops: int = 3) -> Optional[List[str]]:
        """Find path to target region"""
        # BFS for shortest path
        from collections import deque
        
        queue = deque([(['local'], 0)])
        visited = set(['local'])
        
        while queue:
            path, hops = queue.popleft()
            
            if hops > max_hops:
                continue
            
            current = path[-1]
            
            if current == target_region:
                return path
            
            for conn in self.connections:
                if conn['from'] == current and conn['to'] not in visited:
                    visited.add(conn['to'])
                    queue.append((path + [conn['to']], hops + 1))
        
        return None
```

### Multi-Cloud Strategy
```python
class CloudProvider(Enum):
    """Supported cloud providers"""
    AWS = "aws"
    AZURE = "azure"
    GCP = "gcp"
    ON_PREMISES = "on_premises"

class MultiCloudStrategy:
    """Manage applications across multiple clouds"""
    
    def __init__(self):
        self.workload_assignments: Dict[str, CloudProvider] = {}
        self.failover_policies: Dict[str, List[CloudProvider]] = {}
        self.data_residency_rules: Dict[CloudProvider, List[str]] = {}
    
    def assign_workload_to_cloud(self, workload_id: str, provider: CloudProvider):
        """Assign workload to cloud provider"""
        self.workload_assignments[workload_id] = provider
    
    def set_failover_chain(self, workload_id: str, providers: List[CloudProvider]):
        """Define failover order"""
        self.failover_policies[workload_id] = providers
    
    def set_data_residency(self, provider: CloudProvider, allowed_regions: List[str]):
        """Set data residency requirements"""
        self.data_residency_rules[provider] = allowed_regions
    
    def get_active_provider(self, workload_id: str) -> Optional[CloudProvider]:
        """Get active provider for workload"""
        return self.workload_assignments.get(workload_id)
    
    def should_failover(self, workload_id: str, current_provider: CloudProvider, 
                       health_status: Dict) -> bool:
        """Determine if failover needed"""
        
        if health_status.get(current_provider.value, {}).get('healthy', True):
            return False
        
        # Check if other providers available
        failover_chain = self.failover_policies.get(workload_id, [])
        return len(failover_chain) > 1
    
    def get_next_provider(self, workload_id: str, current_provider: CloudProvider) -> Optional[CloudProvider]:
        """Get next provider in failover chain"""
        failover_chain = self.failover_policies.get(workload_id, [])
        
        if current_provider not in failover_chain:
            return None
        
        current_index = failover_chain.index(current_provider)
        
        if current_index < len(failover_chain) - 1:
            return failover_chain[current_index + 1]
        
        return None
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Data Consistency & Synchronization

### Data Replication
```python
class DataReplicationStrategy:
    """Manage data replication across clouds"""
    
    def __init__(self):
        self.replication_rules: List[Dict] = []
        self.sync_status: Dict[str, Dict] = {}
    
    def add_replication_rule(self, data_id: str, source_region: str, target_regions: List[str], 
                           replication_type: str = "async"):
        """Configure data replication"""
        self.replication_rules.append({
            'data_id': data_id,
            'source': source_region,
            'targets': target_regions,
            'type': replication_type,  # sync, async, eventual
            'enabled': True
        })
    
    def get_replication_status(self, data_id: str) -> Dict:
        """Get replication status"""
        for rule in self.replication_rules:
            if rule['data_id'] == data_id:
                return {
                    'data_id': data_id,
                    'source': rule['source'],
                    'targets': rule['targets'],
                    'replicated_to': self.sync_status.get(data_id, {}),
                    'consistent': self._check_consistency(data_id)
                }
        
        return {}
    
    def _check_consistency(self, data_id: str) -> bool:
        """Check if data is consistent across replicas"""
        # Simplified - real implementation checks actual data
        return True
    
    def handle_write_conflict(self, data_id: str, conflicting_writes: List[Dict]) -> Dict:
        """Handle conflicting writes in multi-master replication"""
        
        # Apply last-write-wins strategy (could use vector clocks for better resolution)
        latest = max(conflicting_writes, key=lambda x: x.get('timestamp', 0))
        
        return {
            'winner': latest,
            'losers': [w for w in conflicting_writes if w != latest],
            'resolution_strategy': 'last_write_wins'
        }

class ConsistencyModel:
    """Define consistency guarantees"""
    
    def __init__(self, consistency_level: str):
        self.consistency_level = consistency_level  # strong, eventual, causal
        self.tolerance_level: float = 0.0  # Seconds for eventual
    
    def is_acceptable_lag(self, current_lag_seconds: float) -> bool:
        """Check if replication lag acceptable"""
        if self.consistency_level == "strong":
            return current_lag_seconds < 0.1
        elif self.consistency_level == "causal":
            return current_lag_seconds < 1.0
        else:  # eventual
            return current_lag_seconds < self.tolerance_level
```

### Cross-Cloud Transactions
```python
class DistributedTransaction:
    """Coordinate transactions across clouds"""
    
    def __init__(self, transaction_id: str):
        self.transaction_id = transaction_id
        self.steps: List[Dict] = []
        self.status: str = "pending"
        self.involved_clouds: set = set()
    
    def add_step(self, cloud: str, operation: Dict):
        """Add operation step"""
        self.steps.append({
            'cloud': cloud,
            'operation': operation,
            'status': 'pending'
        })
        self.involved_clouds.add(cloud)
    
    def execute_with_saga_pattern(self) -> bool:
        """Execute transaction using Saga pattern (for eventual consistency)"""
        
        # Forward phase - execute all steps
        for step in self.steps:
            try:
                step['status'] = 'executing'
                # Execute operation
                step['status'] = 'completed'
            except Exception as e:
                step['status'] = 'failed'
                step['error'] = str(e)
                
                # Start compensating transactions
                self._execute_compensation()
                return False
        
        self.status = 'committed'
        return True
    
    def _execute_compensation(self):
        """Rollback failed transaction"""
        # Execute compensating transactions in reverse order
        for step in reversed(self.steps):
            if step['status'] == 'completed':
                print(f"Rolling back operation in {step['cloud']}")
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Management & Orchestration

### Hybrid Cloud Orchestration
```python
class HybridCloudOrchestrator:
    """Orchestrate applications across hybrid environment"""
    
    def __init__(self):
        self.applications: Dict[str, Dict] = {}
        self.deployment_strategies: Dict[str, str] = {}
    
    def deploy_application(self, app_name: str, topology: HybridTopology, placement_strategy: str = "optimal"):
        """Deploy application across hybrid cloud"""
        
        self.applications[app_name] = {
            'topology': topology,
            'placement_strategy': placement_strategy,
            'deployed_at': datetime.now(),
            'status': 'deploying'
        }
    
    def get_application_status(self, app_name: str) -> Dict:
        """Get application status across all regions"""
        
        if app_name not in self.applications:
            return {}
        
        app = self.applications[app_name]
        
        return {
            'app': app_name,
            'topology': app['topology'].topology_name,
            'regions': app['topology'].regions,
            'status': app['status'],
            'last_updated': datetime.now().isoformat()
        }
    
    def handle_region_failure(self, app_name: str, failed_region: str) -> bool:
        """Handle failure of a region"""
        
        if app_name not in self.applications:
            return False
        
        app = self.applications[app_name]
        topology = app['topology']
        
        print(f"⚠️  Region {failed_region} failed for {app_name}")
        
        # Mark region unhealthy
        if failed_region in topology.regions:
            topology.regions[failed_region]['healthy'] = False
        
        # Find alternate regions
        available_regions = [r for r in topology.regions if topology.regions[r].get('healthy', True)]
        
        if available_regions:
            print(f"✅ Failover to: {available_regions[0]}")
            return True
        
        print(f"❌ No alternate regions available")
        return False
```

### Cost Optimization
```python
class HybridCostOptimizer:
    """Optimize costs across hybrid environment"""
    
    def __init__(self):
        self.cost_data: Dict[str, float] = {}
        self.optimization_rules: List[Dict] = []
    
    def record_region_cost(self, region: str, hourly_cost: float):
        """Record cost for region"""
        self.cost_data[region] = hourly_cost
    
    def recommend_cost_optimization(self, workload: Dict) -> List[Dict]:
        """Recommend cost optimizations"""
        
        recommendations = []
        
        # Check if workload could run cheaper elsewhere
        for region, cost in self.cost_data.items():
            if cost < self.cost_data.get(workload.get('current_region', 'local'), float('inf')):
                recommendations.append({
                    'type': 'migrate_region',
                    'from_region': workload.get('current_region'),
                    'to_region': region,
                    'potential_savings_percent': 20
                })
        
        return recommendations
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Hybrid Cloud Architecture Checklist

✅ **Topology & Design**
- [ ] Cloud providers selected
- [ ] Workload classification done
- [ ] Placement strategy defined
- [ ] Failover paths identified
- [ ] Connectivity planned

✅ **Data Management**
- [ ] Replication strategy defined
- [ ] Consistency level set
- [ ] Conflict resolution planned
- [ ] Backup strategy documented
- [ ] RPO/RTO targets set

✅ **Orchestration**
- [ ] Deployment tool selected
- [ ] Multi-cloud framework chosen
- [ ] Health checks configured
- [ ] Failover automation built
- [ ] Monitoring unified

✅ **Cost Management**
- [ ] Cost allocation model
- [ ] Optimization rules set
- [ ] Cloud pricing compared
- [ ] Reserved instance strategy
- [ ] Cost dashboard built

✅ **Operations**
- [ ] Incident response plan
- [ ] On-call rotation
- [ ] Runbooks documented
- [ ] Regular drills scheduled
- [ ] Training completed
</VSCode.Cell>
```